# 🤖 Interview Intelligence Agent
**Autor:** Guilherme Rodrigues | [GitHub](https://github.com/Gbrito90) | [LinkedIn](https://www.linkedin.com/in/guilhermebrito-rodrigues)

---

## O que esse agente faz?

Este agente automatiza três etapas críticas do processo seletivo técnico:

| Etapa | Input | Output |
|-------|-------|--------|
| **1. Análise da Vaga** | PDF com Job Description | Resumo estruturado: stack, senioridade, requisitos técnicos e comportamentais |
| **2. Geração de Perguntas** | Análise da etapa 1 | Perguntas técnicas (por stack) + comportamentais (método STAR) |
| **3. Parecer do Candidato** | Transcript da entrevista | Parecer profissional com score por dimensão e recomendação final |

**Problema que resolve:** Recrutadores técnicos perdem horas preparando entrevistas e escrevendo pareceres. Este agente faz isso em segundos, com qualidade consistente.

---

## ⚙️ Setup

In [ ]:
# Instalar dependências
!pip install openai pymupdf -q

In [ ]:
import fitz  # PyMuPDF
import openai
import os
from IPython.display import Markdown, display

# -------------------------------------------------------------------
# 🔑 Cole sua API Key aqui (ou use variável de ambiente)
# No Colab: vá em Secrets (ícone de cadeado) e adicione OPENAI_API_KEY
# -------------------------------------------------------------------
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv('OPENAI_API_KEY', 'sua-chave-aqui')

client = openai.OpenAI(api_key=api_key)
MODEL = 'gpt-4o'

print('✅ Setup completo!')

---
## 📄 Etapa 1 — Leitura do PDF da Vaga

In [ ]:
def extrair_texto_pdf(caminho_pdf: str) -> str:
    """Extrai o texto de um PDF usando PyMuPDF."""
    doc = fitz.open(caminho_pdf)
    texto = ''
    for pagina in doc:
        texto += pagina.get_text()
    return texto.strip()


# -------------------------------------------------------------------
# OPÇÃO A: Upload de arquivo no Colab
# -------------------------------------------------------------------
from google.colab import files

print('📎 Faça o upload do PDF da Job Description:')
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
texto_vaga = extrair_texto_pdf(nome_arquivo)

print(f'\n✅ PDF lido com sucesso! ({len(texto_vaga)} caracteres)')
print('\n--- Prévia ---')
print(texto_vaga[:500], '...')

---
## 🔍 Etapa 2 — Análise Estruturada da Vaga

In [ ]:
PROMPT_ANALISE_VAGA = """
Você é um especialista em recrutamento técnico de tecnologia com profundo conhecimento em stacks de desenvolvimento, infraestrutura, dados e segurança.

Analise a Job Description abaixo e extraia as informações de forma estruturada.

Retorne EXATAMENTE neste formato Markdown:

## 📋 Análise da Vaga

**Cargo:** [título exato]
**Senioridade:** [Júnior / Pleno / Sênior / Especialista / Lead / Manager]
**Modelo de trabalho:** [Remoto / Híbrido / Presencial]

### 🛠️ Stack Técnica
- **Obrigatório:** [liste as tecnologias mandatórias]
- **Desejável:** [liste as tecnologias desejáveis]

### 🎯 Responsabilidades Principais
[liste as 4-5 principais responsabilidades]

### 🧠 Perfil Comportamental Esperado
[liste 4-5 competências comportamentais implícitas ou explícitas na vaga]

### ⚠️ Pontos de Atenção
[liste 2-3 aspectos críticos que o recrutador deve investigar na entrevista]
"""


def analisar_vaga(texto_jd: str) -> str:
    """Analisa a JD e retorna um resumo estruturado."""
    resposta = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_ANALISE_VAGA},
            {"role": "user", "content": f"Job Description:\n\n{texto_jd}"}
        ],
        temperature=0.3
    )
    return resposta.choices[0].message.content


print('🔍 Analisando a vaga...')
analise_vaga = analisar_vaga(texto_vaga)
display(Markdown(analise_vaga))

---
## ❓ Etapa 3 — Geração de Perguntas de Entrevista

In [ ]:
PROMPT_PERGUNTAS = """
Você é um especialista em recrutamento técnico. Com base na análise da vaga fornecida, crie um roteiro completo de entrevista.

Gere EXATAMENTE neste formato Markdown:

## ❓ Roteiro de Entrevista

### 🔧 Perguntas Técnicas
*(baseadas na stack obrigatória e responsabilidades)*

1. [pergunta técnica direta]
   - 💡 *O que avaliar: [critério de avaliação]*

2. [pergunta técnica situacional]
   - 💡 *O que avaliar: [critério de avaliação]*

[gere 5-6 perguntas técnicas]

### 🧠 Perguntas Comportamentais (Método STAR)
*(baseadas no perfil comportamental esperado)*

1. [pergunta STAR — Situação/Tarefa/Ação/Resultado]
   - 💡 *Competência avaliada: [competência]*

[gere 4-5 perguntas comportamentais]

### 🔎 Perguntas de Investigação
*(para os pontos de atenção identificados na vaga)*

1. [pergunta específica para investigar o ponto crítico]

[gere 2-3 perguntas de investigação]
"""


def gerar_perguntas(analise: str) -> str:
    """Gera o roteiro de perguntas com base na análise da vaga."""
    resposta = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_PERGUNTAS},
            {"role": "user", "content": f"Análise da vaga:\n\n{analise}"}
        ],
        temperature=0.4
    )
    return resposta.choices[0].message.content


print('❓ Gerando roteiro de entrevista...')
roteiro = gerar_perguntas(analise_vaga)
display(Markdown(roteiro))

---
## 📝 Etapa 4 — Cole o Transcript da Entrevista

In [ ]:
# -------------------------------------------------------------------
# Cole aqui o resumo/transcript da sua entrevista com o candidato
# Pode ser um resumo em tópicos, transcrição, ou suas anotações
# -------------------------------------------------------------------

TRANSCRIPT = """
Cole aqui o transcript ou resumo da entrevista.

Exemplo:
- Candidato: João Silva, 5 anos de experiência com Python
- Demonstrou sólido conhecimento em Django e FastAPI
- Quando perguntado sobre microservices, explicou bem o padrão mas não tinha experiência prática com Kubernetes
- Situação de conflito: relatou episódio onde discordou do tech lead sobre arquitetura, resolveu através de POC comparativa
- Pretensão salarial: R$ 15.000
- Disponibilidade: 30 dias
- Motivação para mudança: busca empresa com maior maturidade técnica e cultura de engenharia
"""

print('✅ Transcript pronto para análise!')
print(f'Tamanho: {len(TRANSCRIPT)} caracteres')

---
## ⭐ Etapa 5 — Geração do Parecer Final

In [ ]:
PROMPT_PARECER = """
Você é um especialista em recrutamento técnico sênior com anos de experiência avaliando candidatos para posições de tecnologia.

Com base na análise da vaga e no transcript da entrevista, elabore um parecer técnico-comportamental completo e profissional.

Retorne EXATAMENTE neste formato Markdown:

## ⭐ Parecer do Candidato

**Candidato:** [nome se mencionado, ou 'Candidato Avaliado']
**Vaga:** [título da vaga]
**Data da avaliação:** [hoje]

---

### 📊 Scorecard

| Dimensão | Nota (1-5) | Observação |
|----------|-----------|------------|
| Aderência técnica à stack | ⭐⭐⭐⭐ | [breve justificativa] |
| Profundidade técnica | ⭐⭐⭐ | [breve justificativa] |
| Competências comportamentais | ⭐⭐⭐⭐ | [breve justificativa] |
| Senioridade demonstrada | ⭐⭐⭐ | [breve justificativa] |
| Fit cultural / motivação | ⭐⭐⭐⭐ | [breve justificativa] |

**Score geral:** X/5

---

### ✅ Pontos Fortes
[liste 3-4 pontos fortes com evidências da entrevista]

### ⚠️ Pontos de Atenção
[liste 2-3 pontos de atenção ou gaps identificados]

### 💬 Resumo Executivo
[parágrafo de 3-4 linhas resumindo o candidato para o gestor]

### 🏁 Recomendação
**[ ] AVANÇAR** | **[ ] AVANÇAR COM RESSALVAS** | **[ ] NÃO AVANÇAR**

*Justificativa:* [1-2 linhas explicando a recomendação]
"""


def gerar_parecer(analise: str, transcript: str) -> str:
    """Gera o parecer final do candidato."""
    contexto = f"""Análise da Vaga:\n{analise}\n\n---\n\nTranscript da Entrevista:\n{transcript}"""
    
    resposta = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_PARECER},
            {"role": "user", "content": contexto}
        ],
        temperature=0.3
    )
    return resposta.choices[0].message.content


print('⭐ Gerando parecer do candidato...')
parecer = gerar_parecer(analise_vaga, TRANSCRIPT)
display(Markdown(parecer))

---
## 💾 Exportar Tudo para um Arquivo

In [ ]:
from datetime import datetime

def exportar_relatorio(analise: str, roteiro: str, parecer: str, nome_candidato: str = 'candidato'):
    """Salva todo o relatório em um arquivo Markdown."""
    data = datetime.now().strftime('%Y-%m-%d_%H-%M')
    nome_arquivo = f'relatorio_{nome_candidato}_{data}.md'
    
    conteudo = f"""# 🤖 Interview Intelligence Report
Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M')}

---

{analise}

---

{roteiro}

---

{parecer}
"""
    
    with open(nome_arquivo, 'w', encoding='utf-8') as f:
        f.write(conteudo)
    
    print(f'✅ Relatório salvo: {nome_arquivo}')
    return nome_arquivo


# Altere o nome do candidato
nome = 'joao_silva'

arquivo = exportar_relatorio(analise_vaga, roteiro, parecer, nome)

# Baixar o arquivo no Colab
from google.colab import files
files.download(arquivo)

---

## 📖 Como usar

1. **Abra no Google Colab** — nenhuma instalação local necessária
2. **Configure sua API Key** — use os Secrets do Colab (ícone de cadeado 🔒)
3. **Rode célula por célula** — cada etapa depende da anterior
4. **Faça o upload do PDF** da Job Description na Etapa 1
5. **Cole o transcript** da entrevista na Etapa 4
6. **Exporte o relatório** completo em Markdown

---

## 🏗️ Arquitetura

```
PDF da JD
    ↓ PyMuPDF (extração de texto)
Etapa 1: Análise Estruturada da Vaga
    ↓ GPT-4o (temperature=0.3)
Etapa 2: Roteiro de Perguntas
    ↓ GPT-4o (temperature=0.4)
    ↓ [você faz a entrevista]
Transcript da Entrevista
    ↓
Etapa 3: Parecer Final + Scorecard
    ↓ GPT-4o (temperature=0.3)
Relatório .md exportado
```

---

*Projeto desenvolvido por [Guilherme Rodrigues](https://github.com/Gbrito90) — Tech Recruiter em transição para AI Engineering.*